# Notebook 1: Data Pre-Processing & Exploratory Data Analysis

## Theoretical Foundation
As emphasized in **Section 2.2 (Pre-processing data)** of *Forecasting: theory and practice* by Petropoulos et al., high-quality forecasts depend heavily on data preparation. This notebook implements:
1. **Robust Outlier Handling (Section 2.2.4):** Using Median Absolute Deviation (MAD) to clean the demand signal.
2. **Box-Cox Transformations (Section 2.2.1):** To stabilize variance before any modeling takes place.
3. **Time Series Decomposition (Section 2.2.2):** To visualize trend and seasonality.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import boxcox
from statsmodels.tsa.seasonal import seasonal_decompose

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(14,6), 'font.size':11})

import warnings
warnings.filterwarnings('ignore')

# Load the raw dataset (previously prepared by v2 pipeline)
PANEL_PATH = "../outputs/m5_prepared/m5_monthly_panel.parquet"
df = pd.read_parquet(PANEL_PATH)
df['month'] = pd.to_datetime(df['month'].astype(str))
print(f"Loaded {df.shape[0]} rows. Data ranges from {df['month'].min()} to {df['month'].max()}")

### 1. Robust Outlier Handling
*Reference: Section 2.2.4 - Robust handling of outliers in time series forecasting.*
Outliers can severely distort both classical and ML models. We use the Median Absolute Deviation (MAD) to identify and clip outliers robustly.

In [ ]:
def clip_outliers_mad(series, threshold=3.5):
    median = series.median()
    mad = np.median(np.abs(series - median))
    if mad == 0:
        return series # Avoid division by zero
    modified_z_scores = 0.6745 * (series - median) / mad
    
    # Clip values where modified Z-score > threshold
    upper_bound = median + (threshold * mad / 0.6745)
    lower_bound = max(0, median - (threshold * mad / 0.6745)) # demand >= 0
    
    return series.clip(lower=lower_bound, upper=upper_bound)

# Apply to each series independently
df['demand_clean'] = df.groupby('series_id')['demand'].transform(clip_outliers_mad)

# Visualize the effect on a volatile series
sample_series = df['series_id'].unique()[0]
sample_df = df[df['series_id'] == sample_series].sort_values('month')

plt.figure(figsize=(14, 5))
plt.plot(sample_df['month'], sample_df['demand'], label='Original Demand', alpha=0.5, marker='o')
plt.plot(sample_df['month'], sample_df['demand_clean'], label='Cleaned Demand (MAD)', marker='x')
plt.title(f"Outlier Handling for Series: {sample_series}")
plt.legend()
plt.tight_layout()
plt.savefig('plots/01_outlier_handling.png')
plt.show()

### 2. Box-Cox Transformation
*Reference: Section 2.2.1 - Box-Cox transformations.*
To stabilize variance (especially important for series where variance scales with the mean), we apply a Box-Cox transformation.

In [ ]:
def apply_boxcox(series):
    # Box-Cox requires strictly positive data
    # We add a small constant (e.g., 1) to handle zeros
    transformed, lmbda = boxcox(series + 1)
    return transformed, lmbda

lambdas = {}
transformed_demand = []

for series_id, group in df.groupby('series_id'):
    t, l = apply_boxcox(group['demand_clean'].values)
    lambdas[series_id] = l
    # Assign back to the group
    df.loc[group.index, 'demand_transformed'] = t

print(f"Average Lambda chosen: {np.mean(list(lambdas.values())):.3f}")

plt.figure(figsize=(14, 5))
sns.histplot(list(lambdas.values()), bins=30, kde=True)
plt.title('Distribution of Box-Cox Lambda values across all series')
plt.xlabel('Lambda')
plt.tight_layout()
plt.savefig('plots/02_boxcox_lambdas.png')
plt.show()

### 3. Time Series Decomposition
*Reference: Section 2.2.2 - Time series decomposition.*
We decompose the aggregated total demand to verify the underlying structural components.

In [ ]:
# Aggregate total demand across all series
total_demand = df.groupby('month')['demand_clean'].sum().reset_index()
total_demand.set_index('month', inplace=True)

# Additive decomposition on the original scale
decomp = seasonal_decompose(total_demand['demand_clean'], model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], color='black'); axes[0].set_title('Observed')
decomp.trend.plot(ax=axes[1], color='coral'); axes[1].set_title('Trend')
decomp.seasonal.plot(ax=axes[2], color='seagreen'); axes[2].set_title('Seasonal')
decomp.resid.plot(ax=axes[3], color='grey'); axes[3].set_title('Residual')
plt.suptitle('Seasonal Decomposition of Total Cleaned Demand', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/03_decomposition.png')
plt.show()

In [ ]:
# Save the cleaned and transformed dataset
df.to_parquet('data/01_preprocessed_m5.parquet', index=False)
# Save lambdas for back-transformation later
pd.Series(lambdas).to_csv('data/boxcox_lambdas.csv')
print("Successfully saved preprocessed data.")